# Windowed decoding — accuracy over time, all subjects

The counterpart of `Full_Epoch_Analysis.ipynb`. Where that one asks *"given the whole trial,
how well can it be classified"* — one number per subject, which is what a manuscript reports
— this asks **when** the information is there, sliding a short window across the epoch and
cross-validating at every position.

Every step lives in `src/windowed_batch.py`; this notebook is only the driver. What the two
analyses share lives in `src/analysis_common.py`.
**All settings live in the Parameters cell below** — nothing under it needs editing.

| stage | call | writes |
|---|---|---|
| 1 | `run_windowed_batch()` | `Analysis/<label>/Cache/`, `.../Windowed/Metrics/Individuals/<mode>/`, `.../Windowed/Figures/<subject>/<mode>/` |
| 2 | `summarize_windowed()` | `.../Windowed/Metrics/Group/`, `.../Windowed/Figures/Group/<mode>/`, `.../Windowed/windowed_summary.{csv,json}` |

Outputs live under **`Analysis/<label>/`** — one tree per *class set*, `MH_LH_RH_FR` for the
four-class config, `LH_RH` for a binary one — with both analyses inside it and the epoch cache
shared between them:

```
Analysis/<label>/Cache/        preprocessed epochs, shared with the full-epoch analysis
Analysis/<label>/Windowed/     this notebook's outputs
Analysis/<label>/FullEpoch/    Full_Epoch_Analysis.ipynb's outputs
```

Changing `desired_events` therefore cannot overwrite a previous run's outputs, reuse its cached
epochs, or have `SKIP_DONE` hand back its results. Set `LABEL` in the Parameters cell to
separate runs that share a class set but differ elsewhere (different filters, electrode
group, ...).

`Cache/`, `Figures/` and `Metrics/` are rebuildable and gitignored; the two `windowed_summary.*`
files are the committed artifacts and carry the run's provenance (timestamp, git commit,
package versions, full params).

## What this measures

Class-mean centering is a legitimate part of a covariance/CSP pipeline, but it is
**label-dependent**: you must already know a trial's class to centre it, so it cannot be applied
to an unlabelled trial and the live loop does not apply it. The question is therefore not
"is centering good?" but **"can a classifier trained on centered data still decode live,
uncentered data?"**

So every subject is evaluated over a train/test centering matrix. Modes are named
`<train>2<test>`, where `c` = class means computed **per XDF file** (what `EEG_Preprocessing`
produces with `CenterByClass=True`, so it also removes between-session drift) and `u` = not
centered. The same letters mean the same recipes in `Session_Analysis` and
`Full_Epoch_Analysis`.

|  | test centered | test uncentered |
|---|---|---|
| **train centered** | `c2c` — what `Main_Experiment` measures | **`c2u` — the deployment question** |
| **train uncentered** | `u2c` — symmetry check | `u2u` — a pipeline that never centers |

All four use the same loaded epochs, the same balancing selection and the **same CV folds**, so
the `delta_*` columns are exactly paired per subject:

- `delta_cost_of_centering_at_inference` = `c2c − c2u` — how much of a centered-trained model's
  measured accuracy evaporates once it is served uncentered data.
- `delta_centered_vs_never_centered` = `c2u − u2u` — whether that model still beats one that
  never centered at all. Negative means train uncentered for deployment.

### Read these before the headline number

1. **Pooled-centering limitation.** The centered copy's class means were computed **per XDF
   file**, over every trial of that class in that file — test trials included, which is the
   leak. So in `c2u` the test data is clean uncentered signal but the training data still
   carries it. `c2u` answers *"what if you train the way `Main_Experiment` does today and then
   deploy"*, not *"what if you train centering correctly and then deploy"*. For a leak-free
   number see `Session_Analysis.ipynb` (`cross`/`c2u`) or `Full_Epoch_Analysis.ipynb` (`l2u`).
2. **The pre-cue window must be at chance.** `acc_precue_mean` covers windows entirely before
   the cue. If it is above chance, something is leaking and the headline number is not
   trustworthy — the summary flags this loudly.
3. **Within-subject error bars are anticonservative.** Any two of the 10 CV training sets share
   ~78% of their trials, so fold scores are correlated and `std/√n_folds` understates the
   variance. The columns are named `acc_eval_within_*` for that reason. The claim about the
   pipeline rests on the **between-subject** spread in the `GROUP` row.
4. **The 4-class config runs unbalanced by design.** Balancing applies only when a `'Rest'`
   class is present. So raw accuracy sits against a per-subject `majority_baseline` rather than
   `1/n_classes` — which is why **macro F1** is reported beside it.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings('ignore')

from src.windowed_batch import *

# src.preprocessing forces %matplotlib qt at import time, so set the backend after it.
# The batch itself renders headlessly under Agg regardless.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

print('subjects          :', list_subjects())
print('electrode groups  :', list(ELECTRODE_GROUPS))
print('modes             :', list(MODES))
print('eval window       :', EVAL_WINDOW, '| pre-cue:', PRE_CUE_WINDOW)

subjects          : ['AEH', 'AN', 'AV', 'BA', 'EA', 'EE', 'GS', 'ID', 'IO', 'LD', 'MN', 'NS', 'NZ', 'SK', 'SM', 'SN', 'TA', 'TM']
electrode groups  : ['FP', 'AF', 'F', 'FC', 'C', 'CP', 'P', 'PO', 'O']
modes             : ['c2c', 'u2u', 'c2u', 'u2c']
eval window       : (2.0, 5.0) | pre-cue: (-2.0, 0.0)


## Parameters — the only cell you need to edit

Everything the pipeline reads, spelled out. `default_params()` seeds it, then each key is set
explicitly so the value is visible and editable here rather than in `src/`.

`check_params` rejects a misspelled key (which would otherwise be silently ignored) and the
combinations that fail deep inside preprocessing; `describe_params` marks with `*` whatever
differs from the library default.

**`LABEL` matters more here than anywhere.** The output tree is named from `desired_events`
alone, and `SKIP_DONE` hands back a subject's saved result whenever its *class list* matches —
it does not look at the filter band or the electrode set. So changing a non-class param without
either setting `LABEL` or setting `FORCE=True` **and** `SKIP_DONE=False` would silently mix two
runs in one summary. `check_params` warns when the tree on disk was built with different params.

In [8]:
# ══ PARAMETERS ═══════════════════════════════════════════════════════════════════
ELECTRODE_GROUP_NAMES = 'FC+C+CP+P'   # groups available: printed by the cell above
LABEL = None                          # Analysis/<label>/ tree; None -> named from
                                      # desired_events. Set it when you change any
                                      # other param (see the note above).

PARAMS = default_params(electrode_group_names=ELECTRODE_GROUP_NAMES)

# ── active ───────────────────────────────────────────────────────────────────────
PARAMS['desired_events'] = ['MiddleHand','LeftHand','RightHand' ,'FixatedRest']
PARAMS['PerformAvgRef']  = True     # average re-reference
PARAMS['AddRefChannel']  = False    # reconstruct FCz (the online reference); needs
                                    # PerformAvgRef. False -> no FCz, 35 picks not 36
PARAMS['CenterByClass']  = True     # per-class mean removal (label-dependent, so the
                                    # live loop cannot apply it). HERE the centered /
                                    # uncentered epoch variants override it per build
PARAMS['PerformCsd']     = False    # current-source-density transform
PARAMS['PerformAsr']     = False    # Artifact Subspace Reconstruction on the continuous
                                    # data, BEFORE ICA and before the band-pass. Off =
                                    # every existing result unchanged. Flipping it
                                    # rebuilds the epoch cache; set LABEL too if you
                                    # want the ASR and non-ASR trees side by side
PARAMS['asr_cutoff']     = 20       # rejection threshold, SD of the clean calibration
                                    # data. Lower = more aggressive
PARAMS['asr_max_bad_chans'] = 0.1   # max bad-channel fraction a calibration window may have
PARAMS['asr_backend']    = 'asrpy'  # 'asrpy' (euclid only) or 'meegkit'. meegkit
                                    # must be 0.1.7: newer ones need pyriemann>=0.7,
                                    # and 0.12 needs numpy 2, which breaks mne 1.6.1
PARAMS['asr_method']     = 'euclid' # 'euclid', or 'riemann' (Blum et al. 2019) which
                                    # requires asr_backend='meegkit' - asrpy accepts
                                    # 'riemann' but silently runs euclid, so we reject it
PARAMS['asr_estimator']  = 'lwf'    # meegkit only. 'riemann' REQUIRES a regularising
                                    # estimator: the average reference makes the block
                                    # covariances singular and the riemannian mean needs
                                    # positive definite input ('scm' fails outright)
PARAMS['filter_method']  = 'iir'
PARAMS['LowPass']        = 8        # band-pass low edge, Hz
PARAMS['HighPass']       = 32       # band-pass high edge, Hz
PARAMS['epoch_tmin']     = -5       # epoch crop, s relative to cue
PARAMS['epoch_tmax']     = 6
PARAMS['classifier_window_s'] = 0.2   # training crop, s
PARAMS['classifier_window_e'] = 4
PARAMS['windowed_prediction_params'] = {'win_len': 2, 'win_step': 0.25}  # the slide
PARAMS['augmentation_params']        = {'win_len': 0, 'win_step': 0.25}  # 0 = off
PARAMS['pipeline_name'] = 'ts+FGDA'

# ── read only by the CSP / FBCSP pipelines - inert while pipeline_name is ts+FGDA ─
PARAMS['n_components']       = 8
PARAMS['n_components_fbcsp'] = 8
PARAMS['filters_bands']      = [[7, 12], [12, 20], [20, 28], [28, 35]]

# ── set by the pipeline itself; assigning them here has NO effect ────────────────
#   bad_electrodes              per subject, from get_subject_bad_electrodes
#   events_trigger_dict         per subject, from epochs.event_id
#   epoch_tmins_and_maxes_grid  vestigial - read nowhere

# ── what to run ──────────────────────────────────────────────────────────────────
SUBJECTS  = None            # None = all; or ['BA', 'AEH']
RUN_MODES = list(MODES)     # all four; or ['c2c', 'c2u'] for the centered-trained row
FORCE     = False           # ignore the epoch cache and rebuild
SKIP_DONE = True            # reuse a subject's saved result (resumable runs)
SAVE_FIGS = True

# ─────────────────────────────────────────────────────────────────────────────────
paths = project_paths(label=LABEL, params_dict=PARAMS)
check_params(PARAMS, label=LABEL, paths=paths)
describe_params(PARAMS)

print(f"\nanalysis root : {paths.analysis_root}")
print(f"windowed out  : {paths.out}")
print(f"shared cache  : {paths.cache}")
print()
windowed_status(paths=paths)

  !! 'MH_LH_RH_FR' was built with different params: ['Electorde_Group', 'AddRefChannel'].
     Results merge by subject rather than replace, so this run would mix the two.
     Set LABEL to a new name, or FORCE=True (and SKIP_DONE=False) to rebuild the tree.
ACTIVE
  desired_events               ['MiddleHand', 'LeftHand', 'RightHand', 'FixatedRest']    classes to decode; also names the Analysis/<label>/ tree
  Electorde_Group              36 ch: FC5, FC3, FC1 ... P8                               channel picks, and the order the classifier sees them in
  PerformAvgRef                True                                                      average re-reference
  AddRefChannel              * False                                                     reconstruct FCz (the online reference); requires PerformAvgRef
  CenterByClass                True                                                      per-class mean removal; each epoch variant overrides it per build
  PerformCsd             

,subject,n_files,cached_centered,cached_uncentered,metrics_c2c,metrics_u2u,metrics_c2u,metrics_u2c
0,AEH,3,True,True,False,False,False,False
1,AN,4,True,True,False,False,False,False
2,AV,6,False,False,False,False,False,False
3,BA,3,True,True,False,False,False,False
4,EA,4,True,True,False,False,False,False
5,EE,3,True,True,False,False,False,False
6,ID,3,True,True,False,False,False,False
7,IO,3,False,False,False,False,False,False
8,LD,3,True,True,False,False,False,False
9,MN,4,False,False,False,False,False,False


## Stage 1 — benchmark every subject

For each subject: load and cache **both** epoch variants, assert they are trial-aligned,
balance once (shared selection, so the variants stay aligned), then cross-validate every cell
of the matrix with `RepeatedStratifiedKFold(5, n_repeats=2)` and save the accuracy-over-time
and confusion-matrix figures per mode.

The whole matrix is computed per subject in one pass — that shared state is exactly what makes
the modes comparable.

Resumable and safe to re-run. `SKIP_DONE` reuses a subject's saved result instead of
recomputing it, so an interrupted run picks up where it left off. A subject that fails is
reported and skipped — it never aborts the batch; rerun the failures via `SUBJECTS`.

The epoch cache is keyed on the recordings and on the preprocessing params, so editing a
param in the Parameters cell rebuilds it automatically — `FORCE` is for when you want a rebuild
anyway. Note that the *classifier* params (window, pipeline, prediction slide) are **not** part
of that key: changing one of those needs `SKIP_DONE=False` or a new `LABEL`.

In [9]:
out = run_windowed_batch(SUBJECTS, modes=RUN_MODES, force=FORCE, skip_done=SKIP_DONE,
                         save_figs=SAVE_FIGS, params_dict=PARAMS, paths=paths,
                         label=LABEL)

output label: 'MH_LH_RH_FR'  ->  Analysis\MH_LH_RH_FR\Windowed

[1/16] AEH
Loaded 10 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_c2c.json
Loaded 10 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_u2u.json
Loaded 10 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_c2u.json
Loaded 10 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_u2c.json
  [AEH/c2c] reusing saved result (skip_done=True)
  [AEH/u2u] reusing saved result (skip_done=True)
  [AEH/c2u] reusing saved result (skip_done=True)
  [AEH/u2c] reusing saved result (skip_done=True)

[2/16] AN
Loaded 10 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_c2c.json
Loaded 10 subjects from 

## Stage 2 — group statistics and the summary table

Already run at the end of stage 1. Call it on its own to rebuild the summary and the group
figures **from disk**, without re-classifying anything — useful after editing the reporting
code.

In [10]:
summary_df, payload = summarize_windowed(params_dict=PARAMS, paths=paths, label=LABEL)
summary_df

Loaded 16 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_c2c.json
Loaded 16 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_u2u.json
Loaded 16 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_c2u.json
Loaded 16 subjects from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_LH_RH_FR\Windowed\Metrics\Group\group_windowed_u2c.json
  saved Analysis\MH_LH_RH_FR\Windowed\Figures\Group\c2c\accuracy_over_time_group.png
  saved Analysis\MH_LH_RH_FR\Windowed\Figures\Group\c2c\confusion_group_2.0-5.0s.png
  saved Analysis\MH_LH_RH_FR\Windowed\Figures\Group\u2u\accuracy_over_time_group.png
  saved Analysis\MH_LH_RH_FR\Windowed\Figures\Group\u2u\confusion_group_2.0-5.0s.png
  saved Analysis\MH_LH_RH_FR\Windowed\Figures\Group\c2u\accuracy_over_time_group.png
  saved Analysis\MH_LH_

,subject,status,error,n_files,n_epochs,n_epochs_before,n_channels,dropped_bads,classes,n_classes,...,acc_eval_between_ci95_u2u,f1_macro_between_std_u2u,acc_eval_between_std_c2u,acc_eval_between_sem_c2u,acc_eval_between_ci95_c2u,f1_macro_between_std_c2u,acc_eval_between_std_u2c,acc_eval_between_sem_u2c,acc_eval_between_ci95_u2c,f1_macro_between_std_u2c
0,AEH,ok,,3.0,165.0,NaN,32.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AN,ok,,4.0,220.0,NaN,35.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AV,ok,,6.0,174.0,NaN,35.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BA,ok,,3.0,165.0,NaN,31.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EA,ok,,4.0,206.0,NaN,30.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,EE,ok,,3.0,165.0,NaN,35.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID,ok,,3.0,165.0,NaN,33.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,IO,ok,,3.0,165.0,NaN,35.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,LD,ok,,3.0,165.0,NaN,34.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,MN,ok,,4.0,169.0,NaN,35.0,,MiddleHand|LeftHand|RightHand|FixatedRest,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Outputs

- **`windowed_summary.csv`** — one row per subject with all four modes side by side, plus a
  `GROUP` row. `acc_eval_mean_<mode>` and `f1_macro_eval_<mode>` per mode, and the two `delta_*`
  columns that answer the question.
- **`windowed_summary.json`** — the same rows plus the provenance block and the caveats, so a
  result can always be traced back to the code that produced it.
- **`Windowed/Figures/Group/<mode>/`** — group accuracy-over-time and confusion matrix.
- **`Windowed/Figures/<subject>/<mode>/`** — per-subject equivalents.

Everything except the two summary files is gitignored and can be deleted at any time;
re-running rebuilds it. The epoch cache is shared with `Full_Epoch_Analysis.ipynb`, so deleting
it costs both analyses a re-preprocess.